In [41]:
import torch
import torch.nn as nn
from torch.utils.data import DataLoader, Dataset

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

import os
import math
import string
import unicodedata #as some text is french this is important
from collections import Counter
from pickle import load, dump
from dotenv import load_dotenv
from tqdm import tqdm

load_dotenv()

True

In [17]:
# #We will not be using the tokens generated from Autotokenizer
# from transformers import AutoTokenizer
# fra_tokenizer = AutoTokenizer.from_pretrained("Helsinki-NLP/opus-mt-fr-en")
# eng_tokenizer = AutoTokenizer.from_pretrained("FacebookAI/roberta-base")

# print(f"Special Tokens: {eng_tokenizer.all_special_tokens}")
# print(f"Special IDs: {eng_tokenizer.all_special_ids}")
# print(f"Special Tokens: {fra_tokenizer.all_special_tokens}")
# print(f"Special IDs: {fra_tokenizer.all_special_ids}")

# tokens_file = "data/tokens.pkl" # for jupyter 
# with open(tokens_file, 'rb') as file:
#     tokens = load(file)

In [18]:
PAD = "<pad>"
SOS = "<sos>"
EOS = "<eos>"
UNK = "<unk>"
TABLE = str.maketrans("","",string.punctuation.replace("'","")) # as french words include '

In [32]:
MIN_FREQ = 5
filepath = "data/eng-fra.txt"
vocabpath = "data/vocabs.pkl"

In [20]:
def preprocess(sent): 
    sent = unicodedata.normalize("NFKC", sent)
    return sent.lower().translate(TABLE)

In [33]:
class Vocab():
    def __init__(self, lang, min_freq):
        self.stoi = {PAD:0,SOS:1,EOS:2,UNK:3}
        self.itos = {0:PAD,1:SOS,2:EOS,3:UNK}
        self.freqs = {}
        self.nxt_idx = 4
        self.lang = lang
        self.min_freq = min_freq

    def build_vocab(self, corpus): #we should get the corpus as an iterable of all the words
        freqs = Counter(corpus)
        # VERY IMP: Counter can store in random order for diff runs so sort before vocab.
        self.freqs = dict(sorted(freqs.items(), key=lambda x:x[1], reverse=True)) #higher freq -> lower index
        for word, freq in self.freqs.items():
            if freq >= self.min_freq and word not in self.stoi:
                self.stoi[word] = self.nxt_idx
                self.itos[self.nxt_idx] = word
                self.nxt_idx += 1

    def encode(self, sent):
        tokens = [self.stoi[SOS]]
        tokens += [self.stoi.get(word, self.stoi[UNK]) for word in sent]
        tokens += [self.stoi[EOS]]
        
        return tokens

    def decode(self, tokens):
        return " ".join([self.itos.get(token,UNK) for token in tokens[1:-1]])
        

In [42]:
class Fra2EngDataset(Dataset):
    def __init__(self, filepath, vocab_path=None):
        super().__init__()
        self.fra_vocab = Vocab(lang="fra", min_freq=MIN_FREQ)
        self.eng_vocab = Vocab(lang="eng", min_freq=MIN_FREQ)
        self.pairs = []

        if vocab_path is None:
            with open(filepath, 'r', encoding="utf-8") as file:
                corpus = {"eng": [], "fra": []}
                for line in file:
                    pair = line.strip().split("\t")
                    if len(pair) != 2:
                        continue
                    eng_tokens = preprocess(pair[0]).split()
                    fra_tokens = preprocess(pair[1]).split()
                    corpus["eng"].extend(eng_tokens)
                    corpus["fra"].extend(fra_tokens)
                    self.pairs.append((eng_tokens, fra_tokens))

            self.eng_vocab.build_vocab(corpus["eng"])
            self.fra_vocab.build_vocab(corpus["fra"])

        else:
            with open(vocab_path, 'rb') as f:
                vocabs = load(f)
                self.eng_vocab = vocabs["eng"]
                self.fra_vocab = vocabs["fra"]

    def __len__(self):
        return len(self.pairs)
    
    def __getitem__(self, idx):
        pair = self.pairs[idx]
        eng_tokens = self.eng_vocab.encode(pair[0])
        fra_tokens = self.fra_vocab.encode(pair[1])

        eng_tensor = torch.tensor(eng_tokens, dtype=torch.long)
        fra_tensor = torch.tensor(fra_tokens, dtype=torch.long)

        return fra_tensor, eng_tensor # since fra -> eng

In [46]:
dataset = Fra2EngDataset(filepath)
dataset.eng_vocab.nxt_idx, dataset.fra_vocab.nxt_idx

(5515, 8575)

In [ ]:
# with open(vocabpath, 'wb') as f:
#     dump({"eng":dataset.eng_vocab, "fra":dataset.fra_vocab}, f)
# dataset = Fra2EngDataset(filepath, vocabpath)
# dataset.eng_vocab.nxt_idx, dataset.fra_vocab.nxt_idx

(5515, 8575)